# AIMO3 Day3 — Real Solver 1問実行確認

**目的**: Kaggle 上で real solver を 1問だけロード・実行できるか確認する  
**solver_type**: `real`  
**Day3 スコープ**: model load / 1問実行 / runtime 記録 / 保存物確認。32問完走・改善実験は Day4 以降  

---

## BLOCKED 条件

| 条件 | 発火セル |
|---|---|
| MODEL_PATH が None | Cell 3 |
| model load 失敗 (OOM / missing file 等) | Cell 5 |
| dependency 不足 (bitsandbytes 等) | Cell 5 |
| 1問実行失敗 | Cell 7 |
| output 保存不能 | Cell 8 |

> **禁止**: 32問完走・精度改善・複数モデル比較・public score 議論

In [ ]:
# ============================================================
# Cell 1: 設定・定数
# Day3: solver_type = "real" 固定
# ============================================================
import os

SOLVER_TYPE = "real"  # Day3 は real 固定
assert SOLVER_TYPE == "real", f"Day3 は real のみ許可。現在: {SOLVER_TYPE!r}"

# Dataset path (Day2 で確認済み)
DATASET_PATH = "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3"

# MODEL_PATH_CANDIDATES
# バグ修正: Day2 では第2・第3要素間のカンマ欠落で文字列が連結されていた
# 修正前 (バグ): 
#   "/kaggle/input/models/qwen-lm/qwen-3-5/transformers/qwen3.5-27b/1"
#   "/kaggle/input/nvidia-openmath-nemotron-14b"  ← カンマなしで連結
# 修正後: 各要素にカンマを付与
MODEL_PATH_CANDIDATES = [
    "/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1",
    "/kaggle/input/models/qwen-lm/qwen-3-5/transformers/qwen3.5-27b/1",  # カンマ追加
    "/kaggle/input/nvidia-openmath-nemotron-14b",
    "/kaggle/input/openmath-nemotron-14b-kaggle",
    "/kaggle/input/numina-math-7b-tir",
    "/kaggle/input/deepseek-math-7b-instruct",
    "/kaggle/input/qwen2-5-math-7b-instruct",
    "/kaggle/input/internlm2-math-plus-7b",
]

OUTPUT_DIR    = "/kaggle/working"
LOG_DIR       = os.path.join(OUTPUT_DIR, "logs")
RESULT_PATH   = os.path.join(LOG_DIR, "day3_result.json")

os.makedirs(LOG_DIR, exist_ok=True)

print(f"SOLVER_TYPE  : {SOLVER_TYPE}")
print(f"DATASET_PATH : {DATASET_PATH}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")
print(f"RESULT_PATH  : {RESULT_PATH}")

In [ ]:
# ============================================================
# Cell 2: 環境確認
# ============================================================
import sys, subprocess

print("=" * 50)
print("Python:", sys.version)

import pandas as pd
print(f"pandas       : {pd.__version__}")

import torch
print(f"torch        : {torch.__version__}")
print(f"CUDA avail   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA ver     : {torch.version.cuda}")
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU[{i}]       : {p.name}  ({p.total_memory/1e9:.1f} GB)")
else:
    raise RuntimeError("BLOCKED: CUDA not available")

import transformers
print(f"transformers : {transformers.__version__}")

# bitsandbytes (4-bit quantization for 120B model)
try:
    import bitsandbytes as bnb
    print(f"bitsandbytes : {bnb.__version__}  OK")
    BNB_AVAILABLE = True
except ImportError as e:
    print(f"bitsandbytes : MISSING — {e}")
    print("WARNING: 4-bit quant unavailable. 120B モデルは OOM の可能性あり")
    BNB_AVAILABLE = False

print("=" * 50)

In [ ]:
# ============================================================
# Cell 3: Model path 確認 → MODEL_PATH を 1本決定
# BLOCKED: MODEL_PATH が None のまま
# ============================================================

MODEL_PATH = None

print("=== Model path 候補 ===")
for candidate in MODEL_PATH_CANDIDATES:
    exists = os.path.isdir(candidate)
    marker = "OK" if exists else "--"
    print(f"  [{marker}]  {candidate}")
    if exists and MODEL_PATH is None:
        MODEL_PATH = candidate
        files = sorted(os.listdir(candidate))
        print(f"         → ファイル数: {len(files)}")
        print(f"         → 先頭5件  : {files[:5]}")

print()
if MODEL_PATH is None:
    raise RuntimeError(
        "BLOCKED: model path が見つかりません。\n"
        "Kaggle で Add Data からモデルを追加してください。"
    )

print(f"採用 MODEL_PATH: {MODEL_PATH}")

In [ ]:
# ============================================================
# Cell 4: VRAM 確認
# 120B @ bf16 = ~240GB → 4-bit quant (60GB) が必要
# ============================================================

free_gb  = torch.cuda.mem_get_info()[0] / 1e9
total_gb = torch.cuda.mem_get_info()[1] / 1e9

print(f"VRAM free  : {free_gb:.1f} GB")
print(f"VRAM total : {total_gb:.1f} GB")

# モデルファイルサイズからパラメータ数を推定
import glob as _glob
shard_files = _glob.glob(os.path.join(MODEL_PATH, "*.safetensors"))
total_bytes = sum(os.path.getsize(f) for f in shard_files)
print(f"モデルシャード数  : {len(shard_files)}")
print(f"モデル総サイズ    : {total_bytes/1e9:.1f} GB")
print()
if total_gb < total_bytes / 1e9 * 0.5:  # bf16 で 50% 以下なら 4bit quant 必須
    print("→ VRAM 不足の可能性あり。4-bit quantization を使用します。")
    USE_4BIT = True
else:
    print("→ VRAM 余裕あり。bf16 で試みます。")
    USE_4BIT = False

print(f"USE_4BIT: {USE_4BIT}")

In [ ]:
# ============================================================
# Cell 5: Model load
# BLOCKED: OOM / missing config / dependency 不足
# ============================================================
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"MODEL_PATH  : {MODEL_PATH}")
print(f"USE_4BIT    : {USE_4BIT}")
print(f"SOLVER_TYPE : {SOLVER_TYPE}")
print()

load_start = time.monotonic()
load_error = None

try:
    # tokenizer
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_PATH,
        trust_remote_code=True,
    )
    print(f"  tokenizer OK  vocab_size={tokenizer.vocab_size}")

    # model
    print("Loading model...")
    if USE_4BIT and BNB_AVAILABLE:
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            quantization_config=bnb_cfg,
            device_map="auto",
            trust_remote_code=True,
        )
        load_dtype = "int4(nf4)"
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True,
        )
        load_dtype = "bfloat16"

    model.eval()
    load_elapsed = time.monotonic() - load_start
    model_load_ok = True

    print(f"  model OK  dtype={load_dtype}  load_time={load_elapsed:.1f}s")
    vram_used = torch.cuda.memory_allocated() / 1e9
    print(f"  VRAM used after load: {vram_used:.1f} GB")

except Exception as e:
    load_error   = str(e)
    model_load_ok = False
    load_elapsed = time.monotonic() - load_start
    print(f"BLOCKED: model load 失敗")
    print(f"  error: {load_error}")
    raise RuntimeError(f"BLOCKED: model load failed — {load_error}")

In [ ]:
# ============================================================
# Cell 6: データ読み込み + 1問取り出し
# ============================================================
import pandas as pd

test_df = pd.read_csv(os.path.join(DATASET_PATH, "test.csv"))
ID_COL      = "id"
PROBLEM_COL = "problem"

assert ID_COL in test_df.columns,      f"BLOCKED: '{ID_COL}' not in {list(test_df.columns)}"
assert PROBLEM_COL in test_df.columns, f"BLOCKED: '{PROBLEM_COL}' not in {list(test_df.columns)}"

FIRST_ROW  = test_df.iloc[0]
FIRST_ID   = str(FIRST_ROW[ID_COL])
FIRST_PROB = str(FIRST_ROW[PROBLEM_COL])

print(f"NUM_PROBLEMS : {len(test_df)}")
print(f"対象問題 ID  : {FIRST_ID!r}")
print(f"問題文 (先頭300文字):")
print(f"  {FIRST_PROB[:300]}")

In [ ]:
# ============================================================
# Cell 7: 答え抽出ユーティリティ
# ============================================================
import re

def extract_integer(text: str) -> int:
    """モデル出力から整数を抽出する。
    優先順: 1) \\boxed{N}  2) 最終行が純整数  3) 最後の整数
    """
    # 1. LaTeX \boxed{N}
    boxed = re.findall(r'\\boxed\{(\d+)\}', text)
    if boxed:
        val = int(boxed[-1])
        return val % 1000 if val > 999 else val

    # 2. 最終行が純整数
    for line in reversed(text.strip().splitlines()):
        line = line.strip()
        if re.match(r'^\d+$', line):
            val = int(line)
            return val % 1000 if val > 999 else val

    # 3. テキスト中の最後の整数
    matches = re.findall(r'\b(\d{1,6})\b', text)
    if matches:
        val = int(matches[-1])
        return val % 1000 if val > 999 else val

    raise ValueError(f"No integer found in model output: {text[:200]!r}")


def format_answer_5digit(n: int) -> str:
    """整数を 5桁ゼロ埋め文字列に変換する。"""
    assert 0 <= n <= 99999, f"answer out of range: {n}"
    return f"{n:05d}"


# 動作確認
assert extract_integer(r"The answer is \\boxed{42}.") == 42
assert extract_integer("Step 1...\nStep 2...\n57") == 57
assert extract_integer("Therefore the answer is 123.") == 123
assert format_answer_5digit(31) == "00031"
assert format_answer_5digit(0)  == "00000"
print("extract_integer / format_answer_5digit: PASS")

In [ ]:
# ============================================================
# Cell 8: real solver 定義
# solver_type = "real" を assert で強制
# ============================================================

PROMPT_TEMPLATE = """You are a mathematical competition expert. \
Solve the following problem carefully and step by step.
The answer is a non-negative integer from 0 to 999.
At the very end, write your final answer on a new line as just the number.

Problem:
{problem}

Solution:"""


def solve_real(problem_id: str, problem_text: str) -> dict:
    """Real solver: load model output → extract integer → format 5-digit.

    Returns dict with keys:
        problem_id, raw_output, answer_int, answer_5digit,
        elapsed_sec, exception (None or str)
    """
    assert SOLVER_TYPE == "real", (
        f"BLOCKED: solver_type mismatch. expected 'real', got {SOLVER_TYPE!r}"
    )

    result = {
        "problem_id"   : problem_id,
        "model_path"   : MODEL_PATH,
        "solver_type"  : SOLVER_TYPE,
        "raw_output"   : None,
        "answer_int"   : None,
        "answer_5digit": None,
        "elapsed_sec"  : None,
        "exception"    : None,
    }

    t0 = time.monotonic()
    try:
        prompt = PROMPT_TEMPLATE.format(problem=problem_text)

        # Tokenize
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048,
        ).to(model.device)

        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1024,
                do_sample=False,
                temperature=1.0,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )

        # Decode only generated tokens
        generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
        raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

        result["raw_output"] = raw_text

        # Extract answer
        answer_int = extract_integer(raw_text)
        result["answer_int"]    = answer_int
        result["answer_5digit"] = format_answer_5digit(answer_int)

    except Exception as e:
        result["exception"] = f"{type(e).__name__}: {e}"
        # 例外は握りつぶさない — result に記録して再送出
        result["elapsed_sec"] = round(time.monotonic() - t0, 4)
        raise

    result["elapsed_sec"] = round(time.monotonic() - t0, 4)
    return result

In [ ]:
# ============================================================
# Cell 9: 1問だけ real solver で実行 + timer
# BLOCKED: 例外が発生した場合は握りつぶさず BLOCKED とする
# ============================================================
import json, time

print(f"=== 1問実行 (solver_type={SOLVER_TYPE!r}) ===")
print(f"  problem_id : {FIRST_ID!r}")
print(f"  problem    : {FIRST_PROB[:80]}...")
print()

day3_result = None
exec_start  = time.monotonic()

try:
    day3_result = solve_real(FIRST_ID, FIRST_PROB)
    exec_ok = True
except Exception as e:
    exec_ok = False
    exec_elapsed = time.monotonic() - exec_start
    # 例外情報を保存してから BLOCKED
    day3_result = {
        "problem_id"   : FIRST_ID,
        "model_path"   : MODEL_PATH,
        "solver_type"  : SOLVER_TYPE,
        "raw_output"   : None,
        "answer_int"   : None,
        "answer_5digit": None,
        "elapsed_sec"  : round(exec_elapsed, 4),
        "exception"    : f"{type(e).__name__}: {e}",
    }
    # 例外結果を先に保存
    with open(RESULT_PATH, "w") as f:
        json.dump(day3_result, f, indent=2, ensure_ascii=False)
    print(f"BLOCKED: 1問実行失敗")
    print(f"  exception: {day3_result['exception']}")
    print(f"  result saved to: {RESULT_PATH}")
    raise RuntimeError(f"BLOCKED: solve_real failed — {e}")

print("=== 実行結果 ===")
print(f"  problem_id    : {day3_result['problem_id']}")
print(f"  answer_int    : {day3_result['answer_int']}")
print(f"  answer_5digit : {day3_result['answer_5digit']}")
print(f"  elapsed_sec   : {day3_result['elapsed_sec']}s")
print(f"  exception     : {day3_result['exception']}")
print()
print("=== raw_output (先頭500文字) ===")
print(day3_result["raw_output"][:500] if day3_result["raw_output"] else "(empty)")
print()

# runtime 換算
t = day3_result['elapsed_sec']
print(f"=== runtime 換算 ===")
print(f"  1問          : {t:.1f}s")
print(f"  3問 (dummy)  : {t*3:.1f}s  ({t*3/60:.2f} min)")
print(f"  50問         : {t*50:.1f}s  ({t*50/60:.2f} min)")
print(f"  110問 (本番) : {t*110:.1f}s  ({t*110/60:.2f} min)")
print(f"  Kaggle 上限  : {9*3600}s (9h)")
if t > 0:
    capacity = 9 * 3600 / t
    print(f"  9h で解ける問題数上限: {int(capacity)}問")

In [ ]:
# ============================================================
# Cell 10: 5桁整数形式確認 + 保存物
# BLOCKED: answer_5digit が None / 保存失敗
# ============================================================
import json

# 5桁整数形式の検証
answer_5digit = day3_result["answer_5digit"]
assert answer_5digit is not None, "BLOCKED: answer_5digit が None"
assert len(answer_5digit) == 5,   f"BLOCKED: 5桁でない: {answer_5digit!r}"
assert answer_5digit.isdigit(),   f"BLOCKED: 数字でない: {answer_5digit!r}"
print(f"5桁整数形式: {answer_5digit!r}  OK")

# 実行サマリ保存
with open(RESULT_PATH, "w") as f:
    json.dump(day3_result, f, indent=2, ensure_ascii=False)
print(f"保存: {RESULT_PATH}  ({os.path.getsize(RESULT_PATH)} bytes)")

# 保存内容確認
with open(RESULT_PATH) as f:
    saved = json.load(f)
print()
print("=== 保存内容 ===")
for k, v in saved.items():
    if k == "raw_output" and v:
        print(f"  {k}: (長さ {len(v)} 文字)")
    else:
        print(f"  {k}: {v!r}")

In [ ]:
# ============================================================
# Cell 11: Inference Server セットアップ
# 実際の Kaggle submission で使う API
# Day3 では「接続確認のみ」。serve() は呼ばない。
# ============================================================
import sys
sys.path.insert(0, DATASET_PATH)

try:
    import kaggle_evaluation.aimo_3_inference_server as _aimo_srv
    print(f"kaggle_evaluation import: OK")
    print(f"  module path: {_aimo_srv.__file__}")
    print(f"  attributes : {[a for a in dir(_aimo_srv) if not a.startswith('_')]}")
    INFERENCE_API_AVAILABLE = True
except ImportError as e:
    print(f"kaggle_evaluation import: FAILED — {e}")
    INFERENCE_API_AVAILABLE = False

# predict 関数: inference server 用のラッパー
# → solve_real の結果から answer_int だけ返す
def predict(id_: str, problem: str) -> int:
    """Inference Server 用エントリポイント。
    solver_type=real。例外は握りつぶさない。
    """
    assert SOLVER_TYPE == "real"
    r = solve_real(id_, problem)
    return r["answer_int"]

print()
print(f"predict 関数定義: OK")
print(f"INFERENCE_API_AVAILABLE: {INFERENCE_API_AVAILABLE}")
print()
print("NOTE: Day3 では serve() は呼ばない。")
print("      Day4 以降で全問完走時に以下を実行:")
print("      inference_server = _aimo_srv.AIMO3InferenceServer(predict)")
print("      if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):")
print("          inference_server.serve()")
print("      else:")
print("          inference_server.run_local_gateway((DATASET_PATH,))")

In [ ]:
# ============================================================
# Cell 12: Day3 完了チェック
# 全項目 OK = PASS
# ============================================================
import json

checks = {
    "MODEL_PATH_CANDIDATES_fixed"  : "カンマ修正済み",
    "model_path_decided"           : MODEL_PATH,
    "solver_type"                  : SOLVER_TYPE,
    "model_load_ok"                : model_load_ok,
    "problem_id"                   : day3_result.get("problem_id"),
    "answer_int"                   : day3_result.get("answer_int"),
    "answer_5digit"                : day3_result.get("answer_5digit"),
    "elapsed_sec"                  : day3_result.get("elapsed_sec"),
    "exception"                    : day3_result.get("exception"),
    "result_saved"                 : os.path.isfile(RESULT_PATH),
    "inference_api_available"      : INFERENCE_API_AVAILABLE,
}

print("=== Day3 完了チェック ===")
blocked_items = []
for key, val in checks.items():
    is_ok = (
        val is not None
        and val is not False
        and "BLOCKED" not in str(val)
    )
    # exception は None が正常
    if key == "exception":
        is_ok = (val is None)
    status = "OK" if is_ok else "BLOCKED"
    if status == "BLOCKED":
        blocked_items.append(key)
    print(f"  [{status}]  {key}: {val!r}")

print()
if not blocked_items:
    print("Day3: PASS")
    print("  → model load 成功")
    print("  → real solver 1問実行成功")
    print("  → 5桁整数形式確認済み")
    print("  → 保存物確認済み")
    print("  → placeholder と real の混同なし")
else:
    print(f"Day3: BLOCKED — {blocked_items}")
    raise RuntimeError(f"Day3 BLOCKED: {blocked_items}")